
## Module Overview: Classification & Customer Churn Prediction

This module focuses on solving binary classification problems in machine learning by building, interpreting, and feature-engineering a **Customer Churn Prediction System**.


### About this module

- Course page: https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/main/03-classification
- Focus: binary classification, customer churn prediction, logistic regression, feature engineering, and model evaluation.

### Key Objectives & Workflow:

* **Binary Classification Fundamentals:** Frame churn prediction as a binary classification problem, mapping target variables to boolean values ($0$ or $1$).
* **Exploratory Data Analysis (EDA):** Analyze target variable distribution, compute the global churn rate, and identify data types (numerical vs. categorical).
* **Feature Importance & Selection:** Evaluate feature relevance using **Mutual Information** for categorical variables and correlation coefficients for numerical variables to select predictive features.
* **Categorical Encoding:** Transform non-numeric variables into model-ready numerical format using **One-Hot Encoding** (`DictVectorizer` / Scikit-Learn).
* **Logistic Regression:** Train and apply Logistic Regression models to output prediction probabilities, understanding model weights, bias, and sigmoid activation.
* **Model Interpretation:** Convert raw output probabilities into decision thresholds and interpret how feature weights influence the likelihood of churn.

### What is this module about?

Module 3 is about one of the most common machine learning tasks: **classification**. Instead of predicting a number like house price or fuel consumption, we learn how to predict a **yes/no decision**.

In this module, the example is **customer churn prediction**. A company wants to know which customers are likely to leave. The model looks at customer information such as contract type, monthly charges, tenure, and other features, then predicts whether the customer is likely to **churn** (1) or **stay** (0).

This is called a **binary classification problem** because the target has only two possible outcomes.

### Why this matters

Classification is useful in many real-world cases:

- email spam detection
- fraud detection
- disease diagnosis
- customer retention and churn prediction

The main idea is to train a model from past data so it can recognize patterns and predict future outcomes.

### What the module teaches

This module introduces the workflow for a classification project:

1. Prepare the data and understand the target variable.
2. Explore the dataset and look for patterns in the features.
3. Encode categorical data so a machine learning model can use it.
4. Train a **logistic regression** model, which is one of the standard tools for binary classification.
5. Measure model quality using evaluation metrics and decision thresholds.
6. Interpret the model by understanding which features contribute most to the prediction.



In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'

!wget $data -O data-week-3.csv

--2026-10-07 17:09:17--  https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 977501 (955K) [text/plain]
Saving to: ‘data-week-3.csv’

data-week-3.csv     100%[===================>] 954.59K  --.-KB/s    in 0.02s   

2026-10-07 17:09:18 (41.1 MB/s) - ‘data-week-3.csv’ saved [977501/977501]



In [19]:
df = pd.read_csv('data-week-3.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerid        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   seniorcitizen     7043 non-null   int64  
 3   partner           7043 non-null   str    
 4   dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   phoneservice      7043 non-null   str    
 7   multiplelines     7043 non-null   str    
 8   internetservice   7043 non-null   str    
 9   onlinesecurity    7043 non-null   str    
 10  onlinebackup      7043 non-null   str    
 11  deviceprotection  7043 non-null   str    
 12  techsupport       7043 non-null   str    
 13  streamingtv       7043 non-null   str    
 14  streamingmovies   7043 non-null   str    
 15  contract          7043 non-null   str    
 16  paperlessbilling  7043 non-null   str    
 17  paymen

In [20]:
# Clean column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

#clean categorical columns
categorical_columns = list(df.dtypes[df.dtypes == 'string'].index)

for c in categorical_columns: 
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [21]:
df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [ ]:
# Convert totalcharges to numeric
tc=pd.to_numeric(df.totalcharges, errors='coerce')

tc.isnull().sum()

np.int64(11)

In [ ]:
df[tc.isnull()][['customer']]